# Lab 3a: Local ASR with OpenAI Whisper
**Course:** CIS 531/731 | **Module 3: Obtain**

## Objective
Initialize a local instance of OpenAI's Whisper model to transcribe audio without relying on commercial cloud APIs. 

### Environment Notes
* **Inside DevContainer:** Dependencies (`whisper`, `torch`, `ffmpeg`) are strictly pinned and pre-installed per the course manifest.
* **Running Locally (Outside DevContainer):** You must ensure `ffmpeg` is installed on your OS path (e.g., `sudo apt install ffmpeg` or `brew install ffmpeg`) and run `!pip install openai-whisper torch`.

In [1]:
import sys
import warnings
warnings.filterwarnings("ignore")

try:
    import whisper
    import torch
    print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
except ImportError:
    print("Dependencies missing. If running outside the DevContainer, execute: !pip install openai-whisper torch")

PyTorch Version: 2.2.1+cu121 | CUDA Available: False


### 1. Load the Model
We will load the `base` model. If you have a GPU and sufficient VRAM, you may experiment with `small` or `medium`.

In [2]:
# This will download the weights (~140MB for 'base') on the first run.
model = whisper.load_model("base")
print("Whisper 'base' model loaded successfully into memory.")

100%|███████████████████████████████████████| 139M/139M [00:10<00:00, 13.6MiB/s]


Whisper 'base' model loaded successfully into memory.


### 2. Obtain: Download Source Audio (DevContainer Safe)
Because Docker DevContainers isolate hardware microphones, we will simulate an incoming ASR stream by downloading a public-domain audio archive directly into our workspace.
Target: Wilfred Owen's "Anthem for Doomed Youth" from the WWI Poetry Collection.

In [3]:
import urllib.request
import zipfile
from pathlib import Path
import os

# Define the target ZIP archive and output path
zip_url = "https://archive.org/download/wwi_poetry_1407_librivox/wwi_poetry_1407_librivox_64kb_mp3.zip"
zip_filename = "wwi_poetry.zip"
zip_path = Path.cwd() / zip_filename

print(f"Downloading audio archive to: {zip_path}... (This may take a minute)")

try:
    if not zip_path.exists():
        urllib.request.urlretrieve(zip_url, zip_path)
    print(f"Download complete: {zip_path.stat().st_size / (1024*1024):.2f} MB")
except Exception as e:
    print(f"Download failed: {e}")
    raise

# Extract only the target track
target_track = "wwipoem_51_owen_am_64kb.mp3"
audio_path = Path.cwd() / target_track

print(f"Extracting target track: {target_track}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extract(target_track, path=Path.cwd())

print("Extraction complete. Audio is ready for processing.")

Download complete: 75.13 MB
Extracting target track: wwipoem_51_owen_am_64kb.mp3...
Extraction complete. Audio is ready for processing.


### 3. Transcribe Audio using Whisper
We will pass the extracted file path directly to the Whisper model.
Whisper will automatically handle the audio decoding and downsampling (16kHz) internally.

In [4]:
print("Transcribing audio file...")

# Pass the absolute path of the extracted file to the model
result = model.transcribe(str(audio_path))

print("\n--- TRANSCRIPTION OUTPUT ---")
print(result["text"])
print("----------------------------")

# INTEGRATION REQUIREMENT:
# This exact string (result["text"]) is what you must embed and insert into FAISS for Lab 3b.

Transcribing audio file...

--- TRANSCRIPTION OUTPUT ---
 Anthem for doomed youth by Wilfrere Owen. Red for Librevox.org by Alan Matstone. What passing bells for those who die as cattle? Only the monstrous anger of the guns. Only the stuttering rifles rapid rattle, can pat her out their hasty origins. No mockeries for them, no prayers nor bells. Nor any voice of mourning save the choirs. The shrill, demented choirs of wailing shells. And bugles, calling for them from sad shires. What candles may be held to speed them all? Not in the hands of boys, but in their eyes shall shine the holy glimmers of goodbyes. The power of girls' brow shall be their paw. Their flowers, the tenderness of patient minds. And each slow dusk, a drawing down of blinds. End of Anthem for doomed youth by Wilfrere Owen. This recording is in the public domain.
----------------------------
